# STEP 1B — Full Search-Matrix Evidence Harvest V2 (Fast + Resume-Safe)

This notebook is the **automated harvesting backbone** for the full Step-1 literature review.

It starts from:
- the previously audited 121-query search plan;
- the Step 1A/1A.2 open-discovery results;
- three explicit added literature domains:
  1. adaptive education / intelligent tutoring;
  2. selective exposure / information avoidance;
  3. algorithm awareness / perceived personalisation;
- targeted Step 1A.2 terms folded into their appropriate parent domains.

## Fixed search map

- **153 planned queries**
- **40 domains**
- direct, adjacent, counterevidence, foundational, recent, methodological, replication/boundary, novelty and code/material searches
- both **OpenAlex and PubMed** are executed for every planned query so database routing cannot silently exclude a literature

This notebook does **not** decide which papers are scientifically correct, relevant enough to retain, or what they establish.

It does:
1. execute the complete 153-query automated matrix;
2. record every search and result count;
3. retrieve candidate records from PubMed and OpenAlex;
4. preserve search → retrieval → candidate provenance;
5. deduplicate conservatively by exact DOI or exact title+year;
6. retain database abstracts/topics/MeSH terms;
7. build stratified screening queues rather than one opaque ranking score;
8. preserve dedicated counterevidence, direct, foundational, recent and cross-domain queues;
9. generate a manual-source queue for ACM, IEEE, OSF, publishers and other sources not directly queried by the notebook;
10. audit provenance and coverage;
11. export a single screening package plus CSV mirrors.

### Important scope boundary

OpenAlex/PubMed are the automated discovery backbone. They **do not replace** direct ACM, IEEE, OSF, publisher, citation-chain or primary-record verification. Those unresolved source-specific searches are exported explicitly instead of being silently treated as completed.

Base-plan SHA-256: `bea5f8f952fc1ac7e3980978900a92e490be488066893c1d1a2c0386224ebb4e`  
Expanded-plan SHA-256: `2c35711c41d0b0c750a940f0169c3821f019b0f6764e276d1e5da3213f17b177`

**V2 fixes:** batched Drive persistence, indexed deduplication, completion-validated resume logic, and automatic recovery of partially saved V1 searches.

## 1. Install packages

In [ ]:
!pip -q install pandas numpy requests openpyxl biopython rapidfuzz

## 2. Mount Drive and configure the harvest

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import numpy as np
import requests, re, json, os, time, math, hashlib, zipfile
from Bio import Entrez
from rapidfuzz import fuzz

EMAIL = "your_email@example.com"
if EMAIL == "your_email@example.com":
    EMAIL = input("Enter an email for PubMed/OpenAlex polite API access: ").strip()
if "@" not in EMAIL:
    raise ValueError("A valid email is required.")

Entrez.email = EMAIL

ROOT = Path('/content/drive/MyDrive/Step1_Evidence_Corpus')
ROOT.mkdir(parents=True, exist_ok=True)

STEP1B = ROOT / 'Step1B'
STEP1B.mkdir(parents=True, exist_ok=True)

# Retrieval cap is deliberately equal across search roles to avoid preferential
# harvesting of supportive/direct literature.
MAX_PER_QUERY_PER_DATABASE = 100

# Recent searches are filtered from 2024 onward.
RECENT_START_DATE = "2024-01-01"

# Failed searches rerun automatically on the next Run All.
# Successful searches are skipped unless FORCE_RERUN=True.
FORCE_RERUN = False

# Every successful search contributes its own top-N list to the screening package.
PER_QUERY_SCREEN_N = 25

# Batch persistence: avoids thousands of full Google Drive CSV rewrites.
CHECKPOINT_EVERY = 8

print("Corpus:", ROOT)
print("Step1B:", STEP1B)

## 3. Load the frozen expanded query plan

In [ ]:
FULL_PLAN_RECORDS = json.loads(r'''[{"query_plan_id": "QP0001", "domain": "Bahg paradigm", "search_purpose": "DIRECT", "planned_query": "Bahg personalized recommendation category learning information sampling", "preferred_sources": "publisher;OSF;Google Scholar/Semantic Scholar if used", "priority": "HIGH", "rationale": "Reconstruct source paradigm", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0002", "domain": "Bahg paradigm", "search_purpose": "CODE_MATERIALS", "planned_query": "Bahg category learning recommender OSF code materials", "preferred_sources": "OSF;publisher", "priority": "HIGH", "rationale": "Verify generator/recommender implementation", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0003", "domain": "Bahg paradigm", "search_purpose": "REPLICATION_BOUNDARY", "planned_query": "Bahg category learning replication recommendation sampling", "preferred_sources": "Crossref;OpenAlex;publisher", "priority": "HIGH", "rationale": "Locate replications/follow-ups", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0004", "domain": "closed-loop recommender feedback", "search_purpose": "DIRECT", "planned_query": "recommender system feedback loops user behavior future recommendations", "preferred_sources": "ACM;IEEE;OpenAlex", "priority": "HIGH", "rationale": "Direct feedback-loop literature", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0005", "domain": "closed-loop recommender feedback", "search_purpose": "ADJACENT", "planned_query": "preference pollution amplification recommender exposure diversity", "preferred_sources": "ACM;IEEE;OpenAlex", "priority": "HIGH", "rationale": "Mechanistic adjacent literature", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0006", "domain": "closed-loop recommender feedback", "search_purpose": "COUNTEREVIDENCE", "planned_query": "recommender feedback loops no amplification diversity improvement experiment", "preferred_sources": "ACM;IEEE;OpenAlex", "priority": "MEDIUM", "rationale": "Actively seek boundary/null evidence", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0007", "domain": "closed-loop recommender feedback", "search_purpose": "RECENT", "planned_query": "recommender feedback loop human experiment 2024 2025 2026", "preferred_sources": "ACM;IEEE;OpenAlex", "priority": "HIGH", "rationale": "Recent evidence", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0008", "domain": "human-AI recursive interaction", "search_purpose": "DIRECT", "planned_query": "repeated human AI interaction bias amplification experiment", "preferred_sources": "PubMed;OpenAlex;publisher", "priority": "HIGH", "rationale": "Direct recursive human-AI evidence", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0009", "domain": "human-AI recursive interaction", "search_purpose": "COUNTEREVIDENCE", "planned_query": "human AI repeated interaction debiasing correction no amplification", "preferred_sources": "PubMed;OpenAlex", "priority": "MEDIUM", "rationale": "Boundary/null evidence", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0010", "domain": "human-AI recursive interaction", "search_purpose": "RECENT", "planned_query": "human AI feedback loop experiment 2024 2025 2026", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Recent work", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0011", "domain": "real-world platform experiments", "search_purpose": "DIRECT", "planned_query": "randomized algorithmic feed experiment recommendation platform users", "preferred_sources": "OpenAlex;publisher", "priority": "HIGH", "rationale": "Field RCTs", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0012", "domain": "real-world platform experiments", "search_purpose": "RECENT", "planned_query": "Twitter X YouTube Spotify TikTok randomized recommendation experiment 2024 2025 2026", "preferred_sources": "OpenAlex;publisher", "priority": "HIGH", "rationale": "Current platform evidence", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0013", "domain": "real-world platform experiments", "search_purpose": "COUNTEREVIDENCE", "planned_query": "algorithmic feed experiment null effect attitudes behavior", "preferred_sources": "OpenAlex;publisher", "priority": "MEDIUM", "rationale": "Null/boundary evidence", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0014", "domain": "normative learning under biased environments", "search_purpose": "DIRECT", "planned_query": "category base rate learning unequal category frequency human experiment", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Base-rate adaptation", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0015", "domain": "normative learning under biased environments", "search_purpose": "ADJACENT", "planned_query": "skewed training distribution generalization category learning", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Distribution shift", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0016", "domain": "normative learning under biased environments", "search_purpose": "FOUNDATIONAL", "planned_query": "Bayesian category learning base rates environmental statistics", "preferred_sources": "PubMed;OpenAlex", "priority": "MEDIUM", "rationale": "Normative foundations", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0017", "domain": "normative learning under biased environments", "search_purpose": "COUNTEREVIDENCE", "planned_query": "category learning base rate neglect biased sampling maladaptive", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Limits to rational adaptation", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0018", "domain": "normative learning under biased environments", "search_purpose": "METHODOLOGICAL", "planned_query": "distribution shift covariate shift learning evaluation train test mismatch cognition", "preferred_sources": "OpenAlex", "priority": "MEDIUM", "rationale": "Evaluation framework", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0019", "domain": "active information seeking", "search_purpose": "DIRECT", "planned_query": "active information sampling category learning human", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Direct active category learning", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0020", "domain": "active information seeking", "search_purpose": "ADJACENT", "planned_query": "expected information gain value of information sampling human learning", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Information-value literature", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0021", "domain": "active information seeking", "search_purpose": "ADJACENT", "planned_query": "exploration exploitation information sampling belief learning", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Exploration-exploitation", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0022", "domain": "active information seeking", "search_purpose": "FOUNDATIONAL", "planned_query": "optimal experiment design human hypothesis testing information search", "preferred_sources": "PubMed;OpenAlex", "priority": "MEDIUM", "rationale": "Foundational work", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0023", "domain": "active information seeking", "search_purpose": "COUNTEREVIDENCE", "planned_query": "active sampling suboptimal confirmatory search biased belief", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Maladaptive sampling", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0024", "domain": "active information seeking", "search_purpose": "RECENT", "planned_query": "active information sampling learning 2024 2025 2026", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Recent work", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0025", "domain": "compositional sampling data", "search_purpose": "METHODOLOGICAL", "planned_query": "compositional data psychology proportions simplex log ratio", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Methodology", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0026", "domain": "compositional sampling data", "search_purpose": "METHODOLOGICAL", "planned_query": "ipsative measures reliability profile correlations psychology", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Reliability/interpretation", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0027", "domain": "reciprocal sampling-learning loops", "search_purpose": "DIRECT", "planned_query": "bidirectional information sampling concept learning Braunlich Love SEA", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Direct theory/evidence", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0028", "domain": "reciprocal sampling-learning loops", "search_purpose": "ADJACENT", "planned_query": "belief dependent sampling self confirming beliefs learning", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Adjacent mechanism", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0029", "domain": "reciprocal sampling-learning loops", "search_purpose": "COUNTEREVIDENCE", "planned_query": "exploration prevents sampling bias concept learning", "preferred_sources": "PubMed;OpenAlex", "priority": "MEDIUM", "rationale": "Boundary evidence", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0030", "domain": "category-learning systems", "search_purpose": "FOUNDATIONAL", "planned_query": "category learning rule based information integration exemplar prototype multiple systems review", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Core taxonomy", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0031", "domain": "category-learning systems", "search_purpose": "DIRECT", "planned_query": "classification versus inference observational category learning experiment", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Task-format effects", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0032", "domain": "category-learning systems", "search_purpose": "DIRECT", "planned_query": "feedback changes rule based information integration category learning", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Feedback effects", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0033", "domain": "category-learning systems", "search_purpose": "REPLICATION_BOUNDARY", "planned_query": "category learning strategy switching memorization abstraction boundary conditions", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Strategy heterogeneity", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0034", "domain": "learned attention and feature reweighting", "search_purpose": "DIRECT", "planned_query": "selective attention category learning diagnostic dimensions eye movements", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Direct learned attention", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0035", "domain": "learned attention and feature reweighting", "search_purpose": "DIRECT", "planned_query": "learned inattention category learning extra dimensional shift", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Costs of narrowing", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0036", "domain": "learned attention and feature reweighting", "search_purpose": "RECENT", "planned_query": "feature reweighting category learning memory attention 2024 2025 2026", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Recent work", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0037", "domain": "learned attention and feature reweighting", "search_purpose": "COUNTEREVIDENCE", "planned_query": "selective attention improves category generalization efficient learning", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Adaptive narrowing", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0038", "domain": "measurement reactivity", "search_purpose": "DIRECT", "planned_query": "confidence rating reactivity category learning rule memorization", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Confidence reactivity", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0039", "domain": "measurement reactivity", "search_purpose": "DIRECT", "planned_query": "judgment of learning reactivity category learning", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "JOL reactivity", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0040", "domain": "measurement reactivity", "search_purpose": "FOUNDATIONAL", "planned_query": "testing effect retrieval practice learning meta analysis", "preferred_sources": "PubMed;OpenAlex", "priority": "MEDIUM", "rationale": "Testing reactivity", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0041", "domain": "measurement reactivity", "search_purpose": "REPLICATION_BOUNDARY", "planned_query": "confidence JOL reactivity boundary conditions 2024 2025 2026", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Boundary conditions", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0042", "domain": "temporal learning and accumulation", "search_purpose": "DIRECT", "planned_query": "multi session category learning longitudinal attention strategy", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Temporal trajectories", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0043", "domain": "temporal learning and accumulation", "search_purpose": "ADJACENT", "planned_query": "category learning consolidation spacing saturation strategy switching", "preferred_sources": "PubMed;OpenAlex", "priority": "MEDIUM", "rationale": "Alternative trajectories", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0044", "domain": "temporal learning and accumulation", "search_purpose": "COUNTEREVIDENCE", "planned_query": "nonmonotonic U shaped category learning repeated training", "preferred_sources": "PubMed;OpenAlex", "priority": "MEDIUM", "rationale": "Challenge monotonicity", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0045", "domain": "persistence after algorithm removal", "search_purpose": "DIRECT", "planned_query": "AI induced bias persistence after removal experiment", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Post-AI persistence", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0046", "domain": "persistence after algorithm removal", "search_purpose": "DIRECT", "planned_query": "algorithm feed removal persistent behavior field experiment", "preferred_sources": "OpenAlex;publisher", "priority": "HIGH", "rationale": "Platform persistence", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0047", "domain": "persistence after algorithm removal", "search_purpose": "ADJACENT", "planned_query": "delayed category learning generalization retention 24 hour", "preferred_sources": "PubMed;OpenAlex", "priority": "MEDIUM", "rationale": "Learning retention", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0048", "domain": "persistence after algorithm removal", "search_purpose": "COUNTEREVIDENCE", "planned_query": "AI influence disappears after removal experiment", "preferred_sources": "PubMed;OpenAlex", "priority": "MEDIUM", "rationale": "Decay/null evidence", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0049", "domain": "sequential/adaptive experimental design", "search_purpose": "METHODOLOGICAL", "planned_query": "micro randomized trial proximal causal effects adaptive intervention", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Repeated randomisation", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0050", "domain": "sequential/adaptive experimental design", "search_purpose": "METHODOLOGICAL", "planned_query": "SMART dynamic treatment regime sequential multiple assignment randomized trial", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Adaptive regimes", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0051", "domain": "sequential/adaptive experimental design", "search_purpose": "METHODOLOGICAL", "planned_query": "time varying treatment causal inference state dependent intervention", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Causal estimands", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0052", "domain": "control architectures", "search_purpose": "DIRECT", "planned_query": "active versus yoked category learning information sampling", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Yoked precedent", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0053", "domain": "control architectures", "search_purpose": "ADJACENT", "planned_query": "yoked control replay noncontingent feedback agency learning", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Control limitations", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0054", "domain": "control architectures", "search_purpose": "COUNTEREVIDENCE", "planned_query": "yoked exposure recipient state mismatch learning", "preferred_sources": "PubMed;OpenAlex", "priority": "MEDIUM", "rationale": "State-alignment concern", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0055", "domain": "control architectures", "search_purpose": "METHODOLOGICAL", "planned_query": "matched exposure control active learning agency experimental design", "preferred_sources": "PubMed;OpenAlex", "priority": "MEDIUM", "rationale": "Design framework", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0056", "domain": "static versus dynamic personalisation", "search_purpose": "DIRECT", "planned_query": "static versus dynamic personalization recommender human experiment", "preferred_sources": "ACM;IEEE;OpenAlex", "priority": "HIGH", "rationale": "Direct comparison", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0057", "domain": "static versus dynamic personalisation", "search_purpose": "DIRECT", "planned_query": "frozen personalization adaptive recommendation user study", "preferred_sources": "ACM;IEEE;OpenAlex", "priority": "HIGH", "rationale": "Frozen policy", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0058", "domain": "static versus dynamic personalisation", "search_purpose": "RECENT", "planned_query": "dynamic personalization longitudinal user experiment 2024 2025 2026", "preferred_sources": "ACM;IEEE;OpenAlex", "priority": "HIGH", "rationale": "Recent work", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0059", "domain": "correction/reversal/recovery", "search_purpose": "FOUNDATIONAL", "planned_query": "reversal learning extinction relearning recovery category learning", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Foundational correction", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0060", "domain": "correction/reversal/recovery", "search_purpose": "DIRECT", "planned_query": "learned inattention relevance shift category relearning", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Feature correction", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0061", "domain": "correction/reversal/recovery", "search_purpose": "ADJACENT", "planned_query": "belief correction misinformation persistence correction", "preferred_sources": "PubMed;OpenAlex", "priority": "MEDIUM", "rationale": "Adjacent corrective information", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0062", "domain": "correction/reversal/recovery", "search_purpose": "COUNTEREVIDENCE", "planned_query": "reversal learning spontaneous recovery renewal original memory persists", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Erasure alternatives", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0063", "domain": "transfer", "search_purpose": "FOUNDATIONAL", "planned_query": "Barnett Ceci transfer taxonomy near far transfer", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Transfer taxonomy", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0064", "domain": "transfer", "search_purpose": "DIRECT", "planned_query": "category learning transfer learned attention novel structure", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Category transfer", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0065", "domain": "transfer", "search_purpose": "DIRECT", "planned_query": "learned attention transfer feature weighting context", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Attention transfer", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0066", "domain": "transfer", "search_purpose": "COUNTEREVIDENCE", "planned_query": "category learning transfer context specificity failure", "preferred_sources": "PubMed;OpenAlex", "priority": "MEDIUM", "rationale": "Boundary evidence", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0067", "domain": "computational models of behaviour", "search_purpose": "FOUNDATIONAL", "planned_query": "ALCOVE category learning attention model", "preferred_sources": "PubMed;OpenAlex", "priority": "MEDIUM", "rationale": "Foundational model", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0068", "domain": "computational models of behaviour", "search_purpose": "DIRECT", "planned_query": "SEA information sampling concept learning model", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Sampling-learning model", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0069", "domain": "computational models of behaviour", "search_purpose": "DIRECT", "planned_query": "AARM attention category learning eye tracking model", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Attention model", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0070", "domain": "computational models of behaviour", "search_purpose": "ADJACENT", "planned_query": "Bayesian active learning category sampling model", "preferred_sources": "PubMed;OpenAlex", "priority": "MEDIUM", "rationale": "Alternative models", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0071", "domain": "model and parameter identifiability", "search_purpose": "METHODOLOGICAL", "planned_query": "model recovery parameter recovery computational cognitive modeling", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Recovery methodology", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0072", "domain": "model and parameter identifiability", "search_purpose": "METHODOLOGICAL", "planned_query": "identifiability latent cognitive parameters hierarchical models", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Parameter validity", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0073", "domain": "individual differences/susceptibility", "search_purpose": "DIRECT", "planned_query": "individual differences category learning reliability", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Trait stability", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0074", "domain": "individual differences/susceptibility", "search_purpose": "DIRECT", "planned_query": "susceptibility algorithmic influence individual differences experiment", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Algorithm susceptibility", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0075", "domain": "individual differences/susceptibility", "search_purpose": "METHODOLOGICAL", "planned_query": "treatment effect heterogeneity reliability individual differences", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Moderation reliability", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0076", "domain": "individual differences/susceptibility", "search_purpose": "COUNTEREVIDENCE", "planned_query": "stochastic path dependence apparent individual differences learning", "preferred_sources": "PubMed;OpenAlex", "priority": "MEDIUM", "rationale": "Alternative heterogeneity", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0077", "domain": "statistical/graph learning", "search_purpose": "DIRECT", "planned_query": "statistical learning temporal community neural representation Schapiro", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Relational structure", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0078", "domain": "statistical/graph learning", "search_purpose": "DIRECT", "planned_query": "graph learning network structure neural representation human fMRI", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Graph structure", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0079", "domain": "statistical/graph learning", "search_purpose": "RECENT", "planned_query": "graph learning neural representation 2024 2025 2026", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Recent work", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0080", "domain": "experience-dependent representational warping", "search_purpose": "DIRECT", "planned_query": "category learning perceptual warping neural representation", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Category-induced warping", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0081", "domain": "experience-dependent representational warping", "search_purpose": "ADJACENT", "planned_query": "acquired distinctiveness equivalence neural similarity learning", "preferred_sources": "PubMed;OpenAlex", "priority": "MEDIUM", "rationale": "Similarity change", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0082", "domain": "experience-dependent representational warping", "search_purpose": "DIRECT", "planned_query": "feature sharpening visual cortex category learning", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Feature coding", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0083", "domain": "model-based cognitive neuroscience", "search_purpose": "FOUNDATIONAL", "planned_query": "model based cognitive neuroscience category learning model adjudication", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Methodological foundation", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0084", "domain": "model-based cognitive neuroscience", "search_purpose": "DIRECT", "planned_query": "Mack Preston Love category learning fMRI exemplar prototype model", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Direct precedent", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0085", "domain": "model-based cognitive neuroscience", "search_purpose": "METHODOLOGICAL", "planned_query": "RSA PCM encoding model representational model comparison", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Representational inference", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0086", "domain": "fMRI", "search_purpose": "DIRECT", "planned_query": "fMRI category learning multivariate representation RSA", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Direct modality precedent", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0087", "domain": "fMRI", "search_purpose": "METHODOLOGICAL", "planned_query": "crossnobis crossvalidated Mahalanobis representational similarity reliability", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Distance estimator", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0088", "domain": "fMRI", "search_purpose": "COUNTEREVIDENCE", "planned_query": "task fMRI reliability individual differences meta analysis", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Reliability limits", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0089", "domain": "EEG/ERP", "search_purpose": "DIRECT", "planned_query": "EEG ERP category learning N250 P300 multivariate decoding", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Direct modality precedent", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0090", "domain": "EEG/ERP", "search_purpose": "DIRECT", "planned_query": "time resolved RSA category representation EEG", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Temporal representation", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0091", "domain": "EEG/ERP", "search_purpose": "RECENT", "planned_query": "EEG category learning strategy 2024 2025 2026", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Recent evidence", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0092", "domain": "eye tracking", "search_purpose": "DIRECT", "planned_query": "eye movements category learning diagnostic dimensions Rehder Hoffman", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Direct attention evidence", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0093", "domain": "eye tracking", "search_purpose": "DIRECT", "planned_query": "gaze category learning feedback diagnosticity memory", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Learning dynamics", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0094", "domain": "eye tracking", "search_purpose": "METHODOLOGICAL", "planned_query": "eye tracking reliability category learning attention", "preferred_sources": "PubMed;OpenAlex", "priority": "MEDIUM", "rationale": "Measurement quality", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0095", "domain": "pupillometry", "search_purpose": "DIRECT", "planned_query": "pupil category learning uncertainty prediction error", "preferred_sources": "PubMed;OpenAlex", "priority": "MEDIUM", "rationale": "Direct/near direct", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0096", "domain": "pupillometry", "search_purpose": "ADJACENT", "planned_query": "pupil uncertainty learning rate surprise decision making", "preferred_sources": "PubMed;OpenAlex", "priority": "MEDIUM", "rationale": "Process evidence", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0097", "domain": "pupillometry", "search_purpose": "COUNTEREVIDENCE", "planned_query": "pupillometry specificity luminance arousal confounds review", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Interpretive limitations", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0098", "domain": "fNIRS", "search_purpose": "DIRECT", "planned_query": "fNIRS category learning rule similarity learning", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Direct modality evidence", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0099", "domain": "fNIRS", "search_purpose": "METHODOLOGICAL", "planned_query": "fNIRS depth limitation reliability cognition review", "preferred_sources": "PubMed;OpenAlex", "priority": "MEDIUM", "rationale": "Capabilities/limits", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0100", "domain": "MEG", "search_purpose": "DIRECT", "planned_query": "MEG category learning representation decoding", "preferred_sources": "PubMed;OpenAlex", "priority": "MEDIUM", "rationale": "Direct modality precedent", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0101", "domain": "MEG", "search_purpose": "METHODOLOGICAL", "planned_query": "MEG time resolved category representation source localization", "preferred_sources": "PubMed;OpenAlex", "priority": "MEDIUM", "rationale": "Capabilities", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0102", "domain": "multimodal neuroscience", "search_purpose": "DIRECT", "planned_query": "EEG fNIRS category learning multimodal", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Direct multimodal precedent", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0103", "domain": "multimodal neuroscience", "search_purpose": "ADJACENT", "planned_query": "EEG eye tracking category learning attention", "preferred_sources": "PubMed;OpenAlex", "priority": "MEDIUM", "rationale": "Complementarity", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0104", "domain": "multimodal neuroscience", "search_purpose": "ADJACENT", "planned_query": "fMRI eye tracking category learning representation", "preferred_sources": "PubMed;OpenAlex", "priority": "MEDIUM", "rationale": "Complementarity", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0105", "domain": "neural reliability and power", "search_purpose": "METHODOLOGICAL", "planned_query": "task fMRI reliability meta analysis ICC", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "fMRI reliability", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0106", "domain": "neural reliability and power", "search_purpose": "METHODOLOGICAL", "planned_query": "EEG ERP reliability decoding individual differences", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "EEG reliability", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0107", "domain": "neural reliability and power", "search_purpose": "METHODOLOGICAL", "planned_query": "multivariate neural representation reliability RSA", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Representational reliability", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0108", "domain": "neural reliability and power", "search_purpose": "METHODOLOGICAL", "planned_query": "brain behavior correlation sample size winner curse neuroimaging", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Power/inflation", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0109", "domain": "pilot effects and winner's curse", "search_purpose": "METHODOLOGICAL", "planned_query": "pilot study effect size inflation full trial winner curse", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Pilot inflation", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0110", "domain": "pilot effects and winner's curse", "search_purpose": "METHODOLOGICAL", "planned_query": "selecting largest effect winner's curse psychology", "preferred_sources": "PubMed;OpenAlex", "priority": "HIGH", "rationale": "Selection bias", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0111", "domain": "cross-study adaptive programmes", "search_purpose": "METHODOLOGICAL", "planned_query": "discovery confirmation sequential research program preregistration", "preferred_sources": "PubMed;OpenAlex", "priority": "MEDIUM", "rationale": "Staged programme", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0112", "domain": "cross-study adaptive programmes", "search_purpose": "METHODOLOGICAL", "planned_query": "internal pilot adaptive design effect size selection", "preferred_sources": "PubMed;OpenAlex", "priority": "MEDIUM", "rationale": "Cross-study decisions", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0113", "domain": "online versus lab transportability", "search_purpose": "DIRECT", "planned_query": "online versus laboratory category learning replication", "preferred_sources": "PubMed;OpenAlex", "priority": "MEDIUM", "rationale": "Setting transport", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0114", "domain": "online versus lab transportability", "search_purpose": "ADJACENT", "planned_query": "online cognitive experiment lab replication task context", "preferred_sources": "PubMed;OpenAlex", "priority": "MEDIUM", "rationale": "General transportability", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0115", "domain": "online versus lab transportability", "search_purpose": "METHODOLOGICAL", "planned_query": "scanner task adaptation behavioral paradigm context effects", "preferred_sources": "PubMed;OpenAlex", "priority": "MEDIUM", "rationale": "Neural-setting adaptation", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0116", "domain": "programme feasibility", "search_purpose": "METHODOLOGICAL", "planned_query": "longitudinal online experiment attrition repeated sessions psychology", "preferred_sources": "PubMed;OpenAlex", "priority": "MEDIUM", "rationale": "Participant burden", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0117", "domain": "programme feasibility", "search_purpose": "METHODOLOGICAL", "planned_query": "MRI quality control exclusion attrition cognitive neuroscience", "preferred_sources": "PubMed;OpenAlex", "priority": "MEDIUM", "rationale": "Imaging burden", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0118", "domain": "programme feasibility", "search_purpose": "METHODOLOGICAL", "planned_query": "EEG artifact rejection exclusion participant loss", "preferred_sources": "PubMed;OpenAlex", "priority": "MEDIUM", "rationale": "EEG burden", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0119", "domain": "novelty audit", "search_purpose": "NOVELTY", "planned_query": "behavior contingent recommendation category learning human experiment", "preferred_sources": "OpenAlex;ACM;IEEE;PubMed", "priority": "HIGH", "rationale": "Direct conjunction", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0120", "domain": "novelty audit", "search_purpose": "NOVELTY", "planned_query": "adaptive recommender information sampling category learning persistence", "preferred_sources": "OpenAlex;ACM;IEEE;PubMed", "priority": "HIGH", "rationale": "Direct conjunction", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0121", "domain": "novelty audit", "search_purpose": "NOVELTY", "planned_query": "algorithmic personalization neural category representation learning", "preferred_sources": "OpenAlex;PubMed", "priority": "HIGH", "rationale": "Neural conjunction", "status": "PLANNED", "notes": ""}, {"query_plan_id": "QP0122", "domain": "adaptive education / intelligent tutoring", "search_purpose": "DIRECT", "planned_query": "intelligent tutoring system adaptive learner model content selection human learning experiment", "preferred_sources": "OpenAlex;PubMed;publisher", "priority": "HIGH", "rationale": "State-contingent educational content selection.", "status": "PLANNED", "notes": "Added after Step 1A/1A.2 open-discovery audit"}, {"query_plan_id": "QP0123", "domain": "adaptive education / intelligent tutoring", "search_purpose": "DIRECT", "planned_query": "knowledge tracing adaptive problem selection tutoring learning experiment", "preferred_sources": "OpenAlex;PubMed;publisher", "priority": "HIGH", "rationale": "Learner-state estimation linked to future item selection.", "status": "PLANNED", "notes": "Added after Step 1A/1A.2 open-discovery audit"}, {"query_plan_id": "QP0124", "domain": "adaptive education / intelligent tutoring", "search_purpose": "ADJACENT", "planned_query": "open learner model adaptive scaffolding self-regulated learning", "preferred_sources": "OpenAlex;PubMed;publisher", "priority": "MEDIUM", "rationale": "Learner awareness/control of adaptive learner models.", "status": "PLANNED", "notes": "Added after Step 1A/1A.2 open-discovery audit"}, {"query_plan_id": "QP0125", "domain": "adaptive education / intelligent tutoring", "search_purpose": "ADJACENT", "planned_query": "personalized curriculum sequencing adaptive practice scheduling learner model", "preferred_sources": "OpenAlex;PubMed", "priority": "MEDIUM", "rationale": "Adaptive sequencing adjacent to recommender exposure.", "status": "PLANNED", "notes": "Added after Step 1A/1A.2 open-discovery audit"}, {"query_plan_id": "QP0126", "domain": "adaptive education / intelligent tutoring", "search_purpose": "COUNTEREVIDENCE", "planned_query": "adaptive tutoring personalization no benefit negative effect learner control experiment", "preferred_sources": "OpenAlex;PubMed", "priority": "HIGH", "rationale": "Boundary/null/negative effects of adaptive educational systems.", "status": "PLANNED", "notes": "Added after Step 1A/1A.2 open-discovery audit"}, {"query_plan_id": "QP0127", "domain": "adaptive education / intelligent tutoring", "search_purpose": "RECENT", "planned_query": "adaptive education intelligent tutoring learner model experiment 2024 2025 2026", "preferred_sources": "OpenAlex;PubMed", "priority": "HIGH", "rationale": "Recent human evidence.", "status": "PLANNED", "notes": "Added after Step 1A/1A.2 open-discovery audit"}, {"query_plan_id": "QP0128", "domain": "adaptive education / intelligent tutoring", "search_purpose": "METHODOLOGICAL", "planned_query": "adaptive sequencing learner model evaluation experimental design educational technology", "preferred_sources": "OpenAlex", "priority": "HIGH", "rationale": "Methods for evaluating learner-state-contingent sequencing.", "status": "PLANNED", "notes": "Added after Step 1A/1A.2 open-discovery audit"}, {"query_plan_id": "QP0129", "domain": "selective exposure / information avoidance", "search_purpose": "DIRECT", "planned_query": "information avoidance belief updating learning experiment", "preferred_sources": "OpenAlex;PubMed", "priority": "HIGH", "rationale": "Direct evidence on avoidance as information acquisition.", "status": "PLANNED", "notes": "Added after Step 1A/1A.2 open-discovery audit"}, {"query_plan_id": "QP0130", "domain": "selective exposure / information avoidance", "search_purpose": "DIRECT", "planned_query": "selective exposure confirmation bias information search experiment", "preferred_sources": "OpenAlex;PubMed", "priority": "HIGH", "rationale": "Self-confirming information selection.", "status": "PLANNED", "notes": "Added after Step 1A/1A.2 open-discovery audit"}, {"query_plan_id": "QP0131", "domain": "selective exposure / information avoidance", "search_purpose": "ADJACENT", "planned_query": "positive test strategy hypothesis testing information search learning", "preferred_sources": "OpenAlex;PubMed", "priority": "MEDIUM", "rationale": "Cognitive hypothesis-testing search strategy.", "status": "PLANNED", "notes": "Added after Step 1A/1A.2 open-discovery audit"}, {"query_plan_id": "QP0132", "domain": "selective exposure / information avoidance", "search_purpose": "COUNTEREVIDENCE", "planned_query": "disconfirming information search accuracy motivation exploration belief updating experiment", "preferred_sources": "OpenAlex;PubMed", "priority": "HIGH", "rationale": "Conditions producing corrective/disconfirming sampling.", "status": "PLANNED", "notes": "Added after Step 1A/1A.2 open-discovery audit"}, {"query_plan_id": "QP0133", "domain": "selective exposure / information avoidance", "search_purpose": "FOUNDATIONAL", "planned_query": "selective exposure information avoidance review belief updating decision making", "preferred_sources": "OpenAlex;PubMed", "priority": "MEDIUM", "rationale": "Foundational theory and taxonomy.", "status": "PLANNED", "notes": "Added after Step 1A/1A.2 open-discovery audit"}, {"query_plan_id": "QP0134", "domain": "selective exposure / information avoidance", "search_purpose": "RECENT", "planned_query": "selective exposure information avoidance belief learning 2024 2025 2026", "preferred_sources": "OpenAlex;PubMed", "priority": "HIGH", "rationale": "Recent evidence.", "status": "PLANNED", "notes": "Added after Step 1A/1A.2 open-discovery audit"}, {"query_plan_id": "QP0135", "domain": "selective exposure / information avoidance", "search_purpose": "ADJACENT", "planned_query": "algorithmic recommendation selective exposure information seeking user behavior", "preferred_sources": "OpenAlex;ACM;IEEE", "priority": "MEDIUM", "rationale": "Bridge selective-exposure literature to algorithmic information environments.", "status": "PLANNED", "notes": "Added after Step 1A/1A.2 open-discovery audit"}, {"query_plan_id": "QP0136", "domain": "algorithm awareness / perceived personalisation", "search_purpose": "DIRECT", "planned_query": "perceived personalization algorithm awareness recommendation behavior experiment", "preferred_sources": "OpenAlex;PubMed;ACM;IEEE", "priority": "HIGH", "rationale": "Actual versus perceived personalisation.", "status": "PLANNED", "notes": "Added after Step 1A/1A.2 open-discovery audit"}, {"query_plan_id": "QP0137", "domain": "algorithm awareness / perceived personalisation", "search_purpose": "DIRECT", "planned_query": "personalization cues perceived algorithm responsiveness user behavior experiment", "preferred_sources": "OpenAlex;ACM;IEEE", "priority": "HIGH", "rationale": "Perceived contingency as a causal pathway.", "status": "PLANNED", "notes": "Added after Step 1A/1A.2 open-discovery audit"}, {"query_plan_id": "QP0138", "domain": "algorithm awareness / perceived personalisation", "search_purpose": "ADJACENT", "planned_query": "algorithm transparency user mental model recommendation personalization behavior", "preferred_sources": "OpenAlex;ACM;IEEE", "priority": "MEDIUM", "rationale": "Algorithm mental models and awareness.", "status": "PLANNED", "notes": "Added after Step 1A/1A.2 open-discovery audit"}, {"query_plan_id": "QP0139", "domain": "algorithm awareness / perceived personalisation", "search_purpose": "COUNTEREVIDENCE", "planned_query": "algorithm awareness personalization no effect behavior experiment", "preferred_sources": "OpenAlex;ACM;IEEE", "priority": "HIGH", "rationale": "Null/boundary evidence.", "status": "PLANNED", "notes": "Added after Step 1A/1A.2 open-discovery audit"}, {"query_plan_id": "QP0140", "domain": "algorithm awareness / perceived personalisation", "search_purpose": "RECENT", "planned_query": "algorithm awareness perceived personalization recommendation 2024 2025 2026", "preferred_sources": "OpenAlex;ACM;IEEE", "priority": "HIGH", "rationale": "Recent evidence.", "status": "PLANNED", "notes": "Added after Step 1A/1A.2 open-discovery audit"}, {"query_plan_id": "QP0141", "domain": "algorithm awareness / perceived personalisation", "search_purpose": "METHODOLOGICAL", "planned_query": "manipulation perceived personalization actual personalization experiment", "preferred_sources": "OpenAlex;ACM;IEEE", "priority": "HIGH", "rationale": "How perceived and actual personalisation are separated.", "status": "PLANNED", "notes": "Added after Step 1A/1A.2 open-discovery audit"}, {"query_plan_id": "QP0142", "domain": "human-AI recursive interaction", "search_purpose": "ADJACENT", "planned_query": "interactive machine learning human feedback adaptive system behavior", "preferred_sources": "OpenAlex;ACM;IEEE", "priority": "MEDIUM", "rationale": "Human feedback changes adaptive system state.", "status": "PLANNED", "notes": "Added after Step 1A/1A.2 open-discovery audit"}, {"query_plan_id": "QP0143", "domain": "human-AI recursive interaction", "search_purpose": "ADJACENT", "planned_query": "hybrid human AI regulation adaptive learning self-regulated learning", "preferred_sources": "OpenAlex", "priority": "MEDIUM", "rationale": "Distributed regulation between human and system.", "status": "PLANNED", "notes": "Added after Step 1A/1A.2 open-discovery audit"}, {"query_plan_id": "QP0144", "domain": "closed-loop recommender feedback", "search_purpose": "ADJACENT", "planned_query": "performative prediction feedback distribution shift decision system", "preferred_sources": "OpenAlex", "priority": "MEDIUM", "rationale": "Formal endogenous distribution-shift analogue.", "status": "PLANNED", "notes": "Added after Step 1A/1A.2 open-discovery audit"}, {"query_plan_id": "QP0145", "domain": "closed-loop recommender feedback", "search_purpose": "ADJACENT", "planned_query": "decision dependent distribution automated decision feedback loop", "preferred_sources": "OpenAlex", "priority": "MEDIUM", "rationale": "Alternative formal vocabulary for feedback.", "status": "PLANNED", "notes": "Added after Step 1A/1A.2 open-discovery audit"}, {"query_plan_id": "QP0146", "domain": "sequential/adaptive experimental design", "search_purpose": "METHODOLOGICAL", "planned_query": "just in time adaptive intervention decision points tailoring variables proximal outcomes", "preferred_sources": "OpenAlex;PubMed", "priority": "HIGH", "rationale": "Formal adaptive-intervention terminology.", "status": "PLANNED", "notes": "Added after Step 1A/1A.2 open-discovery audit"}, {"query_plan_id": "QP0147", "domain": "sequential/adaptive experimental design", "search_purpose": "METHODOLOGICAL", "planned_query": "micro randomized trial tailoring variable proximal effect time varying intervention", "preferred_sources": "OpenAlex;PubMed", "priority": "HIGH", "rationale": "Time-varying causal component identification.", "status": "PLANNED", "notes": "Added after Step 1A/1A.2 open-discovery audit"}, {"query_plan_id": "QP0148", "domain": "control architectures", "search_purpose": "ADJACENT", "planned_query": "self generation choice agency active passive learning memory experiment", "preferred_sources": "OpenAlex;PubMed", "priority": "MEDIUM", "rationale": "Agency/self-generation as rival to contingency.", "status": "PLANNED", "notes": "Added after Step 1A/1A.2 open-discovery audit"}, {"query_plan_id": "QP0149", "domain": "control architectures", "search_purpose": "ADJACENT", "planned_query": "volitional learning self directed acquisition passive exposure memory", "preferred_sources": "OpenAlex;PubMed", "priority": "MEDIUM", "rationale": "Volitional acquisition as a rival pathway.", "status": "PLANNED", "notes": "Added after Step 1A/1A.2 open-discovery audit"}, {"query_plan_id": "QP0150", "domain": "measurement reactivity", "search_purpose": "ADJACENT", "planned_query": "computerized adaptive testing item selection learning testing effect", "preferred_sources": "OpenAlex;PubMed", "priority": "MEDIUM", "rationale": "Adaptive assessment intersecting with testing reactivity.", "status": "PLANNED", "notes": "Added after Step 1A/1A.2 open-discovery audit"}, {"query_plan_id": "QP0151", "domain": "measurement reactivity", "search_purpose": "ADJACENT", "planned_query": "adaptive assessment learner state item sequencing measurement reactivity", "preferred_sources": "OpenAlex;PubMed", "priority": "MEDIUM", "rationale": "State-contingent probes/assessment.", "status": "PLANNED", "notes": "Added after Step 1A/1A.2 open-discovery audit"}, {"query_plan_id": "QP0152", "domain": "measurement reactivity", "search_purpose": "COUNTEREVIDENCE", "planned_query": "confidence rating judgment of learning no reactivity null effect learning experiment", "preferred_sources": "OpenAlex;PubMed", "priority": "HIGH", "rationale": "Null/boundary evidence for measurement reactivity.", "status": "PLANNED", "notes": "Added after Step 1A/1A.2 open-discovery audit"}, {"query_plan_id": "QP0153", "domain": "static versus dynamic personalisation", "search_purpose": "COUNTEREVIDENCE", "planned_query": "static personalization dynamic personalization no difference adaptive recommendation experiment", "preferred_sources": "OpenAlex;ACM;IEEE", "priority": "HIGH", "rationale": "Null evidence for incremental ongoing adaptation.", "status": "PLANNED", "notes": "Added after Step 1A/1A.2 open-discovery audit"}]''')
plan = pd.DataFrame(FULL_PLAN_RECORDS)

EXPECTED_BASE_HASH = "bea5f8f952fc1ac7e3980978900a92e490be488066893c1d1a2c0386224ebb4e"
EXPECTED_FULL_HASH = "2c35711c41d0b0c750a940f0169c3821f019b0f6764e276d1e5da3213f17b177"

CANONICAL_COLS = [
    "query_plan_id","domain","search_purpose","planned_query",
    "preferred_sources","priority","rationale"
]

def plan_hash(df):
    return hashlib.sha256(
        df[CANONICAL_COLS].to_csv(index=False).encode()
    ).hexdigest()

display(plan.head())
print("Queries:", len(plan))
print("Domains:", plan["domain"].nunique())
print("Expanded plan hash:", plan_hash(plan))

## 4. Preflight logic audit

This audit runs **before any search**.

It is intentionally structural. It checks that:
- the original 121-query plan has not been lost;
- the expanded map has 153 unique queries and 40 domains;
- no exact duplicate query strings exist;
- controlled search purposes are valid;
- all three Step-1A additions contain direct, counterevidence and recent searches;
- load-bearing directional domains are not represented only by supportive searches;
- search-source information is present;
- the frozen plan hash matches the audited build.

If any assertion fails, the harvest stops.

In [ ]:
ALLOWED_PURPOSES = {
    "DIRECT","ADJACENT","COUNTEREVIDENCE","FOUNDATIONAL","RECENT",
    "METHODOLOGICAL","REPLICATION_BOUNDARY","NOVELTY","CODE_MATERIALS"
}

NEW_DOMAINS = {
    "adaptive education / intelligent tutoring",
    "selective exposure / information avoidance",
    "algorithm awareness / perceived personalisation"
}

DIRECTIONAL_DOMAINS_REQUIRING_CHALLENGE_SEARCH = {
    "closed-loop recommender feedback",
    "human-AI recursive interaction",
    "real-world platform experiments",
    "normative learning under biased environments",
    "active information seeking",
    "reciprocal sampling-learning loops",
    "learned attention and feature reweighting",
    "measurement reactivity",
    "temporal learning and accumulation",
    "persistence after algorithm removal",
    "control architectures",
    "static versus dynamic personalisation",
    "correction/reversal/recovery",
    "transfer",
    "adaptive education / intelligent tutoring",
    "selective exposure / information avoidance",
    "algorithm awareness / perceived personalisation"
}

def preflight_audit(plan):
    issues = []

    if len(plan) != 153:
        issues.append(f"Expected 153 queries, found {len(plan)}.")
    if plan["domain"].nunique() != 40:
        issues.append(f"Expected 40 domains, found {plan['domain'].nunique()}.")
    if not plan["query_plan_id"].is_unique:
        issues.append("Duplicate query_plan_id values.")

    missing_base = [
        f"QP{i:04d}" for i in range(1,122)
        if f"QP{i:04d}" not in set(plan["query_plan_id"])
    ]
    if missing_base:
        issues.append("Missing original audited query IDs: " + ", ".join(missing_base))

    bad_purposes = sorted(set(plan["search_purpose"]) - ALLOWED_PURPOSES)
    if bad_purposes:
        issues.append("Invalid search purposes: " + ", ".join(bad_purposes))

    empty_queries = plan[plan["planned_query"].astype(str).str.strip()==""]["query_plan_id"].tolist()
    if empty_queries:
        issues.append("Blank planned queries: " + ", ".join(empty_queries))

    empty_sources = plan[plan["preferred_sources"].astype(str).str.strip()==""]["query_plan_id"].tolist()
    if empty_sources:
        issues.append("Blank preferred_sources: " + ", ".join(empty_sources))

    norm_q = (
        plan["planned_query"].astype(str)
        .str.lower()
        .str.replace(r"\s+"," ",regex=True)
        .str.strip()
    )
    dup_q = plan[norm_q.duplicated(keep=False)][["query_plan_id","planned_query"]]
    if len(dup_q):
        issues.append("Exact duplicate planned queries detected: " + "; ".join(dup_q["query_plan_id"].tolist()))

    for d in NEW_DOMAINS:
        x = plan[plan["domain"]==d]
        roles = set(x["search_purpose"])
        if len(x) < 5:
            issues.append(f"{d}: fewer than 5 planned searches.")
        for required in {"DIRECT","COUNTEREVIDENCE","RECENT"}:
            if required not in roles:
                issues.append(f"{d}: missing {required} search.")

    for d in DIRECTIONAL_DOMAINS_REQUIRING_CHALLENGE_SEARCH:
        roles = set(plan.loc[plan["domain"]==d,"search_purpose"])
        if not ({"COUNTEREVIDENCE","REPLICATION_BOUNDARY"} & roles):
            issues.append(f"{d}: no counterevidence or replication/boundary search.")

    actual_hash = plan_hash(plan)
    if actual_hash != EXPECTED_FULL_HASH:
        issues.append(
            "Expanded query-plan hash mismatch. "
            f"Expected {EXPECTED_FULL_HASH}, found {actual_hash}."
        )

    return pd.DataFrame({"issue":issues})

preflight_issues = preflight_audit(plan)
if len(preflight_issues):
    display(preflight_issues)
    raise RuntimeError("Preflight audit failed. Do not harvest until resolved.")

print("PRE-FLIGHT AUDIT: PASS")
print(plan["search_purpose"].value_counts())
print("\nQueries per domain:")
display(plan.groupby("domain").size().rename("planned_queries").reset_index())

## 5. Persistent Step-1B tables

Step 1B search/retrieval tables are isolated from Step 1A so repeated runs do not contaminate earlier discovery records.

The common `candidates.csv` is reused only to avoid creating duplicate candidate IDs across phases.

In [ ]:
CANDIDATE_COLUMNS = [
    "candidate_id","title","authors","year","doi","url",
    "screening_decision","exclusion_reason","duplicate_of",
    "linked_publication_id","notes"
]

SEARCH_COLUMNS = [
    "search_id","query_plan_id","domain","search_purpose","database",
    "date_searched","query_used","filters_used",
    "total_results_reported","results_retrieved",
    "status","error","partial_retrieval","run_timestamp"
]

RETRIEVAL_COLUMNS = [
    "retrieval_id","search_id","query_plan_id","candidate_id",
    "database","database_record_id","retrieval_rank","run_timestamp"
]

SOURCE_RECORD_COLUMNS = [
    "source_record_id","candidate_id","database","database_record_id",
    "title","authors","year","doi","url","abstract",
    "subject_terms","source_topics","publication_type",
    "cited_by_count","retraction_or_correction_flag",
    "retrieved_date"
]

ORIGIN_COLUMNS = [
    "candidate_id","was_preexisting","created_in_step1b","created_date"
]

def load_csv(path, columns):
    if path.exists():
        df = pd.read_csv(path, dtype=str).fillna("")
        for c in columns:
            if c not in df.columns:
                df[c] = ""
        return df[columns]
    df = pd.DataFrame(columns=columns)
    df.to_csv(path, index=False)
    return df

CANDIDATE_PATH = ROOT / "candidates.csv"
SEARCH_PATH = STEP1B / "step1b_searches.csv"
RETRIEVAL_PATH = STEP1B / "step1b_retrievals.csv"
SOURCE_PATH = STEP1B / "step1b_source_records.csv"
ORIGIN_PATH = STEP1B / "step1b_candidate_origin.csv"
PLAN_PATH = STEP1B / "step1b_query_plan.csv"

candidates = load_csv(CANDIDATE_PATH, CANDIDATE_COLUMNS)
searches = load_csv(SEARCH_PATH, SEARCH_COLUMNS)
retrievals = load_csv(RETRIEVAL_PATH, RETRIEVAL_COLUMNS)
source_records = load_csv(SOURCE_PATH, SOURCE_RECORD_COLUMNS)
candidate_origin = load_csv(ORIGIN_PATH, ORIGIN_COLUMNS)

plan.to_csv(PLAN_PATH, index=False)

print("Existing common candidates:", len(candidates))
print("Existing Step1B successful searches:", int((searches["status"]=="OK").sum()) if len(searches) else 0)

## 6. IDs, normalization and safe persistence

In [ ]:
def _max_numeric(series):
    nums = []
    for v in series.astype(str):
        m = re.search(r"(\d+)$", v)
        if m:
            nums.append(int(m.group(1)))
    return max(nums, default=0)

_id_counters = {
    "candidate": _max_numeric(candidates["candidate_id"]),
    "search": _max_numeric(searches["search_id"]),
    "retrieval": _max_numeric(retrievals["retrieval_id"]),
    "source": _max_numeric(source_records["source_record_id"])
}

def _next(kind, prefix, width):
    _id_counters[kind] += 1
    return f"{prefix}{_id_counters[kind]:0{width}d}"

def next_candidate_id(): return _next("candidate","C",6)
def next_search_id(): return _next("search","B1S",6)
def next_retrieval_id(): return _next("retrieval","B1R",7)
def next_source_record_id(): return _next("source","B1SR",7)

def normalize_doi(x):
    x = str(x or "").strip().lower()
    x = re.sub(r"^https?://(dx\.)?doi\.org/","",x)
    x = re.sub(r"^doi:\s*","",x)
    return x.strip()

def normalize_title(x):
    return re.sub(r"[^a-z0-9]+"," ",str(x or "").lower()).strip()

def clean_year(x):
    m = re.search(r"(19|20)\d{2}", str(x or ""))
    return m.group(0) if m else ""

def safe_text(x):
    return "" if x is None else str(x)

def strip_recent_year_tokens(q):
    q = re.sub(r"\b2024\b|\b2025\b|\b2026\b"," ",str(q))
    q = re.sub(r"\s+"," ",q).strip()
    return q

# Fast candidate lookup indices.
doi_index = {}
title_index = {}

def rebuild_candidate_indices():
    global doi_index, title_index
    doi_index = {}
    title_index = {}
    for _, r in candidates.iterrows():
        cid = str(r["candidate_id"])
        doi = normalize_doi(r["doi"])
        nt = normalize_title(r["title"])
        yr = clean_year(r["year"])
        if doi and doi not in doi_index:
            doi_index[doi] = cid
        if nt:
            title_index.setdefault(nt, []).append((cid, yr))

rebuild_candidate_indices()

source_record_keys = {
    (str(r["candidate_id"]), str(r["database"]), str(r["database_record_id"]))
    for _, r in source_records.iterrows()
}
retrieval_keys = {
    (str(r["search_id"]), str(r["candidate_id"]), str(r["database_record_id"]))
    for _, r in retrievals.iterrows()
}

_completed_since_checkpoint = 0

def save_all_core(force=False):
    """Persist full working tables in batches, not once per record."""
    global _completed_since_checkpoint
    if not force and _completed_since_checkpoint < CHECKPOINT_EVERY:
        return
    candidates.to_csv(CANDIDATE_PATH, index=False)
    searches.to_csv(SEARCH_PATH, index=False)
    retrievals.to_csv(RETRIEVAL_PATH, index=False)
    source_records.to_csv(SOURCE_PATH, index=False)
    candidate_origin.to_csv(ORIGIN_PATH, index=False)
    _completed_since_checkpoint = 0
    print("  [Drive checkpoint saved]")

def checkpoint_completed_search():
    global _completed_since_checkpoint
    _completed_since_checkpoint += 1
    save_all_core(force=False)

def repair_incomplete_ok_searches():
    """
    V1 could mark a search OK before all retrieval rows were written.
    Reclassify any such row so it is automatically rerun.
    """
    global searches
    repaired = []
    counts = retrievals.groupby("search_id").size().to_dict() if len(retrievals) else {}
    for i, r in searches.iterrows():
        if str(r["status"]) != "OK":
            continue
        try:
            expected = int(float(r["results_retrieved"]))
        except Exception:
            expected = 0
        actual = int(counts.get(str(r["search_id"]), 0))
        if actual < expected:
            searches.at[i, "status"] = "INCOMPLETE"
            searches.at[i, "error"] = f"Recovered V1 partial write: {actual}/{expected} retrievals persisted."
            repaired.append((r["search_id"], actual, expected))
    if repaired:
        searches.to_csv(SEARCH_PATH,index=False)
        print("Recovered incomplete V1 searches:", len(repaired))
    return repaired

repaired_v1_searches = repair_incomplete_ok_searches()
save_all_core(force=True)

## 7. Conservative candidate deduplication

Automatic merging occurs only when:
1. DOI is exactly the same after normalization; or
2. normalized title is exactly the same **and** publication year is the same (or one year is missing).

Fuzzy title similarity is **never** used to merge candidates automatically. It is exported later for manual review.

In [ ]:
def find_existing_candidate(title, doi="", year=""):
    doi = normalize_doi(doi)
    year = clean_year(year)

    if doi and doi in doi_index:
        return doi_index[doi]

    nt = normalize_title(title)
    if nt and nt in title_index:
        matches = title_index[nt]
        if not year:
            return matches[0][0]
        for cid, yr in matches:
            if yr == year:
                return cid
        for cid, yr in matches:
            if not yr:
                return cid
    return None

def get_or_create_candidate(record):
    global candidates, candidate_origin

    title = safe_text(record.get("title","")).strip()
    doi = normalize_doi(record.get("doi",""))
    year = clean_year(record.get("year",""))

    existing = find_existing_candidate(title, doi, year)
    if existing:
        if existing not in set(candidate_origin["candidate_id"].astype(str)):
            candidate_origin = pd.concat([
                candidate_origin,
                pd.DataFrame([{
                    "candidate_id":existing,
                    "was_preexisting":"yes",
                    "created_in_step1b":"no",
                    "created_date":""
                }])
            ], ignore_index=True)
        return existing

    cid = next_candidate_id()
    row = {
        "candidate_id":cid,
        "title":title,
        "authors":safe_text(record.get("authors","")),
        "year":year,
        "doi":doi,
        "url":safe_text(record.get("url","")),
        "screening_decision":"NOT_SCREENED",
        "exclusion_reason":"",
        "duplicate_of":"",
        "linked_publication_id":"",
        "notes":"Automatically harvested in Step 1B; no scientific screening decision made."
    }
    candidates = pd.concat([candidates,pd.DataFrame([row])],ignore_index=True)
    candidate_origin = pd.concat([
        candidate_origin,
        pd.DataFrame([{
            "candidate_id":cid,
            "was_preexisting":"no",
            "created_in_step1b":"yes",
            "created_date":pd.Timestamp.now().date().isoformat()
        }])
    ], ignore_index=True)

    if doi:
        doi_index.setdefault(doi,cid)
    nt = normalize_title(title)
    if nt:
        title_index.setdefault(nt,[]).append((cid,year))
    return cid

## 8. PubMed retrieval

In [ ]:
def pubmed_query_for(row):
    q = str(row["planned_query"])
    if row["search_purpose"] == "RECENT":
        q = strip_recent_year_tokens(q)
        current = pd.Timestamp.now().date().isoformat().replace("-","/")
        q = f'({q}) AND ("2024/01/01"[Date - Publication] : "{current}"[Date - Publication])'
    return q

def pubmed_search_records(query, retmax=100, retries=3):
    last_error = None

    for attempt in range(retries):
        try:
            h = Entrez.esearch(
                db="pubmed",
                term=query,
                retmax=retmax,
                sort="relevance"
            )
            sr = Entrez.read(h)
            ids = list(sr.get("IdList",[]))
            total = int(sr.get("Count",0))

            if not ids:
                return total, pd.DataFrame()

            rows = []
            for start in range(0,len(ids),100):
                batch = ids[start:start+100]
                time.sleep(0.38)
                h = Entrez.efetch(
                    db="pubmed",
                    id=",".join(batch),
                    rettype="xml"
                )
                data = Entrez.read(h)

                for pa in data.get("PubmedArticle",[]):
                    med = pa["MedlineCitation"]
                    art = med["Article"]

                    pmid = safe_text(med.get("PMID",""))
                    title = safe_text(art.get("ArticleTitle",""))

                    abstract_parts = []
                    abstract_obj = art.get("Abstract",{})
                    for part in abstract_obj.get("AbstractText",[]) if abstract_obj else []:
                        label = getattr(part,"attributes",{}).get("Label","")
                        abstract_parts.append(
                            (label + ": " if label else "") + safe_text(part)
                        )

                    authors = []
                    for a in art.get("AuthorList",[]) or []:
                        collective = safe_text(a.get("CollectiveName",""))
                        if collective:
                            authors.append(collective)
                        else:
                            nm = " ".join(
                                x for x in [
                                    safe_text(a.get("LastName","")),
                                    safe_text(a.get("Initials",""))
                                ] if x
                            )
                            if nm:
                                authors.append(nm)

                    doi = ""
                    for aid in pa.get("PubmedData",{}).get("ArticleIdList",[]):
                        if getattr(aid,"attributes",{}).get("IdType") == "doi":
                            doi = normalize_doi(aid)
                            break

                    ji = art.get("Journal",{}).get("JournalIssue",{})
                    pubdate = ji.get("PubDate",{}) if ji else {}
                    year = safe_text(pubdate.get("Year",""))
                    if not year:
                        year = clean_year(pubdate.get("MedlineDate",""))

                    mesh = []
                    for mh in med.get("MeshHeadingList",[]) or []:
                        d = safe_text(mh.get("DescriptorName",""))
                        if d:
                            mesh.append(d)

                    ptypes = [
                        safe_text(x)
                        for x in art.get("PublicationTypeList",[]) or []
                    ]
                    ptype_text = "; ".join(ptypes)
                    retract_flag = "yes" if any(
                        key.lower() in ptype_text.lower()
                        for key in [
                            "Retracted Publication",
                            "Retraction of Publication",
                            "Published Erratum",
                            "Corrected and Republished Article"
                        ]
                    ) else "no"

                    rows.append({
                        "database":"PubMed",
                        "database_record_id":pmid,
                        "title":title,
                        "authors":"; ".join(authors),
                        "year":clean_year(year),
                        "doi":doi,
                        "url":f"https://pubmed.ncbi.nlm.nih.gov/{pmid}/" if pmid else "",
                        "abstract":" ".join(abstract_parts),
                        "subject_terms":"; ".join(sorted(set(mesh))),
                        "source_topics":"",
                        "publication_type":ptype_text,
                        "cited_by_count":"",
                        "retraction_or_correction_flag":retract_flag
                    })

            return total, pd.DataFrame(rows)

        except Exception as e:
            last_error = e
            time.sleep(1.5 * (2 ** attempt))

    raise last_error

## 9. OpenAlex retrieval

In [ ]:
def reconstruct_openalex_abstract(inv):
    if not inv:
        return ""
    max_pos = max(
        (max(v) for v in inv.values() if v),
        default=-1
    )
    if max_pos < 0:
        return ""
    words = [""] * (max_pos + 1)
    for word, positions in inv.items():
        for pos in positions:
            if 0 <= pos < len(words):
                words[pos] = word
    return " ".join(w for w in words if w)

def openalex_params_for(row, cursor, per_page):
    q = str(row["planned_query"])
    params = {
        "search": strip_recent_year_tokens(q) if row["search_purpose"]=="RECENT" else q,
        "per-page": per_page,
        "cursor": cursor,
        "mailto": EMAIL
    }
    if row["search_purpose"] == "RECENT":
        today = pd.Timestamp.now().date().isoformat()
        params["filter"] = f"from_publication_date:{RECENT_START_DATE},to_publication_date:{today}"
    return params

def openalex_search_records(row, max_records=100, retries=3):
    rows = []
    cursor = "*"
    total = None

    while len(rows) < max_records:
        per_page = min(100, max_records-len(rows))
        params = openalex_params_for(row,cursor,per_page)

        last_error = None
        data = None
        for attempt in range(retries):
            try:
                r = requests.get(
                    "https://api.openalex.org/works",
                    params=params,
                    timeout=45
                )
                if r.status_code == 429 or 500 <= r.status_code < 600:
                    raise requests.HTTPError(
                        f"HTTP {r.status_code}: {r.text[:200]}"
                    )
                r.raise_for_status()
                data = r.json()
                break
            except Exception as e:
                last_error = e
                time.sleep(1.5 * (2 ** attempt))

        if data is None:
            raise last_error

        if total is None:
            total = int(data.get("meta",{}).get("count",0) or 0)

        batch = data.get("results",[])
        if not batch:
            break

        for w in batch:
            title = safe_text(w.get("display_name","")).strip()
            if not title:
                continue

            authors = [
                (a.get("author") or {}).get("display_name","")
                for a in (w.get("authorships",[]) or [])
            ]
            topics = [
                t.get("display_name","")
                for t in (w.get("topics",[]) or [])[:12]
                if t.get("display_name")
            ]
            keywords = [
                k.get("display_name","")
                for k in (w.get("keywords",[]) or [])[:25]
                if k.get("display_name")
            ]

            rows.append({
                "database":"OpenAlex",
                "database_record_id":safe_text(w.get("id","")),
                "title":title,
                "authors":"; ".join(a for a in authors if a),
                "year":clean_year(w.get("publication_year","")),
                "doi":normalize_doi(w.get("doi","")),
                "url":safe_text(
                    (w.get("primary_location") or {}).get("landing_page_url","")
                ),
                "abstract":reconstruct_openalex_abstract(
                    w.get("abstract_inverted_index")
                ),
                "subject_terms":"; ".join(keywords),
                "source_topics":"; ".join(topics),
                "publication_type":safe_text(w.get("type","")),
                "cited_by_count":safe_text(w.get("cited_by_count","")),
                "retraction_or_correction_flag":"yes" if w.get("is_retracted") else "no"
            })

            if len(rows) >= max_records:
                break

        cursor = data.get("meta",{}).get("next_cursor")
        if not cursor:
            break

        time.sleep(0.18)

    return int(total or 0), pd.DataFrame(rows)

## 10. Search execution and provenance storage

**V2 restart/performance logic**

Candidate, retrieval and source-record rows are accumulated in memory. The lightweight search-status table is saved immediately, while the large working tables are checkpointed only every `CHECKPOINT_EVERY` completed searches.

A hard runtime stop can therefore lose only the uncheckpointed batches. On the next run, any `OK` search whose persisted retrieval count is short is automatically reclassified as `INCOMPLETE` and rerun. This avoids thousands of large Google Drive rewrites without silently losing search coverage.

In [ ]:
def _retrieval_count_for_search(search_id):
    if not len(retrievals):
        return 0
    return int((retrievals["search_id"].astype(str)==str(search_id)).sum())

def successful_search_exists(qid, database):
    if FORCE_RERUN or len(searches)==0:
        return False
    x = searches[
        (searches["query_plan_id"].astype(str)==str(qid)) &
        (searches["database"].astype(str)==str(database)) &
        (searches["status"].astype(str)=="OK")
    ].copy()
    if not len(x):
        return False
    x["_ts"] = pd.to_datetime(x["run_timestamp"],errors="coerce")
    x = x.sort_values("_ts",ascending=False)
    for _, r in x.iterrows():
        try:
            expected = int(float(r["results_retrieved"]))
        except Exception:
            expected = 0
        if _retrieval_count_for_search(r["search_id"]) >= expected:
            return True
    return False

def begin_search_log(row, database, query_used, total, n, filters=""):
    global searches
    sid = next_search_id()
    try:
        partial = "yes" if int(total) > int(n) else "no"
    except Exception:
        partial = ""
    rec = {
        "search_id":sid,
        "query_plan_id":row["query_plan_id"],
        "domain":row["domain"],
        "search_purpose":row["search_purpose"],
        "database":database,
        "date_searched":pd.Timestamp.now().date().isoformat(),
        "query_used":query_used,
        "filters_used":filters,
        "total_results_reported":safe_text(total),
        "results_retrieved":safe_text(n),
        "status":"RUNNING",
        "error":"",
        "partial_retrieval":partial,
        "run_timestamp":pd.Timestamp.now().isoformat()
    }
    searches = pd.concat([searches,pd.DataFrame([rec])],ignore_index=True)
    searches.to_csv(SEARCH_PATH,index=False)
    return sid

def mark_search_error(search_id, error):
    global searches
    mask = searches["search_id"].astype(str)==str(search_id)
    searches.loc[mask,"status"] = "ERROR"
    searches.loc[mask,"error"] = safe_text(error)
    searches.to_csv(SEARCH_PATH,index=False)

def add_retrieval(search_id, row, candidate_id, rank):
    global retrievals
    record_id = safe_text(row.get("database_record_id",""))
    key = (str(search_id),str(candidate_id),record_id)
    if key in retrieval_keys:
        return
    rec = {
        "retrieval_id":next_retrieval_id(),
        "search_id":search_id,
        "query_plan_id":searches.loc[
            searches["search_id"].astype(str)==str(search_id),"query_plan_id"
        ].iloc[0],
        "candidate_id":candidate_id,
        "database":safe_text(row.get("database","")),
        "database_record_id":record_id,
        "retrieval_rank":safe_text(rank),
        "run_timestamp":pd.Timestamp.now().isoformat()
    }
    retrievals = pd.concat([retrievals,pd.DataFrame([rec])],ignore_index=True)
    retrieval_keys.add(key)

def add_source_record(row, candidate_id):
    global source_records
    database = safe_text(row.get("database",""))
    record_id = safe_text(row.get("database_record_id",""))
    key = (str(candidate_id),database,record_id)
    if key in source_record_keys:
        return
    rec = {
        "source_record_id":next_source_record_id(),
        "candidate_id":candidate_id,
        "database":database,
        "database_record_id":record_id,
        "title":row.get("title",""),
        "authors":row.get("authors",""),
        "year":row.get("year",""),
        "doi":normalize_doi(row.get("doi","")),
        "url":row.get("url",""),
        "abstract":row.get("abstract",""),
        "subject_terms":row.get("subject_terms",""),
        "source_topics":row.get("source_topics",""),
        "publication_type":row.get("publication_type",""),
        "cited_by_count":row.get("cited_by_count",""),
        "retraction_or_correction_flag":row.get("retraction_or_correction_flag",""),
        "retrieved_date":pd.Timestamp.now().date().isoformat()
    }
    source_records = pd.concat([source_records,pd.DataFrame([rec])],ignore_index=True)
    source_record_keys.add(key)

def process_hits(search_id, hits):
    for rank, (_, rec) in enumerate(hits.iterrows(), start=1):
        cid = get_or_create_candidate(rec)
        add_retrieval(search_id,rec,cid,rank)
        add_source_record(rec,cid)

def finalize_search_ok(search_id):
    global searches
    # Persist only the lightweight search-status table immediately.
    # Large tables are checkpointed every CHECKPOINT_EVERY completed searches.
    # If runtime stops before the next checkpoint, startup validation detects
    # the short retrieval count and reruns those searches automatically.
    mask = searches["search_id"].astype(str)==str(search_id)
    searches.loc[mask,"status"] = "OK"
    searches.loc[mask,"error"] = ""
    searches.to_csv(SEARCH_PATH,index=False)
    checkpoint_completed_search()


## 11. RUN THE COMPLETE AUTOMATED STEP-1B MATRIX

This cell is what makes `Runtime → Run all` conduct the automated harvest.

For each of the 153 planned queries it executes:
- OpenAlex;
- PubMed.

That is 306 query/database executions when starting from scratch.

The same retrieval cap is applied to every search role. Large search spaces are **flagged as partial**, not treated as complete.

In [ ]:
run_rows = []

for idx, row in plan.iterrows():
    qid = row["query_plan_id"]

    # OpenAlex
    if successful_search_exists(qid,"OpenAlex"):
        print("SKIP",qid,"OpenAlex — complete")
    else:
        print("\nRUN",qid,"OpenAlex ::",row["planned_query"])
        sid = None
        try:
            total,hits = openalex_search_records(row,max_records=MAX_PER_QUERY_PER_DATABASE)
            q_used = strip_recent_year_tokens(row["planned_query"]) if row["search_purpose"]=="RECENT" else row["planned_query"]
            filter_used = f"publication_date>={RECENT_START_DATE}" if row["search_purpose"]=="RECENT" else ""
            sid = begin_search_log(row,"OpenAlex",q_used,total,len(hits),filters=filter_used)
            process_hits(sid,hits)
            finalize_search_ok(sid)
            run_rows.append({"query_plan_id":qid,"database":"OpenAlex","total":total,"retrieved":len(hits),"status":"OK"})
            print("  total:",total,"retrieved:",len(hits))
        except Exception as e:
            if sid is not None:
                mark_search_error(sid,e)
            run_rows.append({"query_plan_id":qid,"database":"OpenAlex","total":"","retrieved":0,"status":"ERROR: "+str(e)})
            print("  ERROR:",e)

    time.sleep(0.20)

    # PubMed
    if successful_search_exists(qid,"PubMed"):
        print("SKIP",qid,"PubMed — complete")
    else:
        pm_query = pubmed_query_for(row)
        print("RUN",qid,"PubMed ::",pm_query)
        sid = None
        try:
            total,hits = pubmed_search_records(pm_query,retmax=MAX_PER_QUERY_PER_DATABASE)
            filters_used = f"publication_date>={RECENT_START_DATE}" if row["search_purpose"]=="RECENT" else ""
            sid = begin_search_log(row,"PubMed",pm_query,total,len(hits),filters=filters_used)
            process_hits(sid,hits)
            finalize_search_ok(sid)
            run_rows.append({"query_plan_id":qid,"database":"PubMed","total":total,"retrieved":len(hits),"status":"OK"})
            print("  total:",total,"retrieved:",len(hits))
        except Exception as e:
            if sid is not None:
                mark_search_error(sid,e)
            run_rows.append({"query_plan_id":qid,"database":"PubMed","total":"","retrieved":0,"status":"ERROR: "+str(e)})
            print("  ERROR:",e)

    time.sleep(0.30)

save_all_core(force=True)

run_summary = pd.DataFrame(run_rows)
display(run_summary.tail(30))
print("\nAutomated matrix execution finished.")

## 12. Use only Step-1B successful search executions for downstream reports

If a query was force-rerun, the most recent successful execution for that query/database is used. This prevents duplicate historical runs from inflating candidate counts.

In [ ]:
def latest_successful_searches():
    ok = searches[searches["status"]=="OK"].copy()
    if not len(ok):
        return ok

    valid = []
    for _, r in ok.iterrows():
        try:
            expected = int(float(r["results_retrieved"]))
        except Exception:
            expected = 0
        actual = _retrieval_count_for_search(r["search_id"])
        if actual >= expected:
            valid.append(r)

    if not valid:
        return ok.iloc[0:0].copy()

    ok = pd.DataFrame(valid)
    ok["_ts"] = pd.to_datetime(ok["run_timestamp"],errors="coerce")
    ok = ok.sort_values("_ts")
    ok = ok.groupby(["query_plan_id","database"],as_index=False).tail(1)
    return ok.drop(columns=["_ts"])

active_searches = latest_successful_searches()
active_search_ids = set(active_searches["search_id"].astype(str))

active_retrievals = retrievals[
    retrievals["search_id"].astype(str).isin(active_search_ids)
].copy()

active_candidate_ids = set(active_retrievals["candidate_id"].astype(str))
active_candidates = candidates[
    candidates["candidate_id"].astype(str).isin(active_candidate_ids)
].copy()

active_source_records = source_records[
    source_records["candidate_id"].astype(str).isin(active_candidate_ids)
].copy()

print("Active complete searches:",len(active_searches))
print("Expected complete searches:",len(plan)*2)
print("Step1B retrievals:",len(active_retrievals))
print("Unique Step1B candidates:",len(active_candidates))

## 13. Candidate summary — descriptive, not a scientific score

The notebook does **not** create a single relevance score.

Instead each candidate records:
- domains in which it was retrieved;
- search-purpose strata;
- database coverage;
- number of independent planned queries retrieving it;
- best retrieval ranks;
- whether an abstract is available;
- maximum OpenAlex citation count;
- neutral flags such as DIRECT_HIT, COUNTEREVIDENCE_HIT or CROSS_DOMAIN.

This prevents one global ranking from suppressing counterevidence, niche domains or recent work.

In [ ]:
def numeric_or_nan(x):
    try:
        return float(x)
    except Exception:
        return np.nan

def candidate_summary_table():
    if not len(active_retrievals):
        return pd.DataFrame()

    x = active_retrievals.merge(
        active_searches[
            ["search_id","query_plan_id","domain","search_purpose","database"]
        ],
        on=["search_id","query_plan_id","database"],
        how="left"
    )

    src = active_source_records.copy()
    src["_cites"] = src["cited_by_count"].map(numeric_or_nan)

    rows = []
    for cid,g in x.groupby("candidate_id"):
        c = active_candidates[
            active_candidates["candidate_id"].astype(str)==str(cid)
        ]
        if not len(c):
            continue
        c = c.iloc[0]

        purposes = sorted(set(g["search_purpose"].astype(str)))
        domains = sorted(set(g["domain"].astype(str)))
        dbs = sorted(set(g["database"].astype(str)))

        ranks = pd.to_numeric(g["retrieval_rank"],errors="coerce")
        best_rank = int(ranks.min()) if ranks.notna().any() else ""

        sg = src[src["candidate_id"].astype(str)==str(cid)]
        max_cites = (
            int(sg["_cites"].max())
            if len(sg) and sg["_cites"].notna().any()
            else ""
        )
        has_abs = "yes" if (
            len(sg) and sg["abstract"].astype(str).str.strip().ne("").any()
        ) else "no"

        flags = []
        for p in purposes:
            flags.append(p+"_HIT")
        if len(domains) >= 2:
            flags.append("CROSS_DOMAIN")
        if g["query_plan_id"].nunique() >= 2:
            flags.append("MULTI_QUERY")
        if len(dbs) >= 2:
            flags.append("MULTI_DATABASE")

        rows.append({
            "candidate_id":cid,
            "title":c["title"],
            "authors":c["authors"],
            "year":c["year"],
            "doi":c["doi"],
            "url":c["url"],
            "screening_decision":c["screening_decision"],
            "domain_count":len(domains),
            "domains":"; ".join(domains),
            "query_count":g["query_plan_id"].nunique(),
            "search_purposes":"; ".join(purposes),
            "database_count":len(dbs),
            "databases":"; ".join(dbs),
            "best_retrieval_rank":best_rank,
            "max_openalex_cited_by":max_cites,
            "abstract_available":has_abs,
            "flags":"; ".join(flags),
            "manual_relevance":"",
            "manual_exclusion_reason":"",
            "manual_notes":""
        })

    out = pd.DataFrame(rows)
    if len(out):
        out["_year"] = pd.to_numeric(out["year"],errors="coerce")
        out = out.sort_values(
            ["domain_count","query_count","best_retrieval_rank","_year"],
            ascending=[False,False,True,False],
            na_position="last"
        ).drop(columns=["_year"])
    return out

candidate_summary = candidate_summary_table()
display(candidate_summary.head(50))

## 14. Stratified screening queue

This is the main anti-narrowing screening structure.

A paper can appear separately within:
- a domain;
- a search-purpose stratum.

Therefore a globally popular paper cannot crowd out:
- counterevidence;
- recent work;
- foundational work;
- methodology;
- replication/boundary-condition literature.

In [ ]:
def stratified_screening_table():
    if not len(active_retrievals):
        return pd.DataFrame()

    x = active_retrievals.merge(
        active_searches[
            ["search_id","query_plan_id","domain","search_purpose","database"]
        ],
        on=["search_id","query_plan_id","database"],
        how="left"
    ).merge(
        active_candidates[
            ["candidate_id","title","year","doi","screening_decision"]
        ],
        on="candidate_id",
        how="left"
    )

    rows = []
    for keys,g in x.groupby(
        ["candidate_id","domain","search_purpose"],
        dropna=False
    ):
        cid,domain,purpose = keys
        ranks = pd.to_numeric(g["retrieval_rank"],errors="coerce")
        rows.append({
            "candidate_id":cid,
            "domain":domain,
            "search_purpose":purpose,
            "title":g.iloc[0]["title"],
            "year":g.iloc[0]["year"],
            "doi":g.iloc[0]["doi"],
            "screening_decision":g.iloc[0]["screening_decision"],
            "query_count_in_stratum":g["query_plan_id"].nunique(),
            "databases":"; ".join(sorted(set(g["database"].astype(str)))),
            "best_rank_in_stratum":int(ranks.min()) if ranks.notna().any() else "",
            "manual_relevance":"",
            "manual_notes":""
        })

    out = pd.DataFrame(rows)
    if len(out):
        out = out.sort_values(
            ["domain","search_purpose","best_rank_in_stratum","candidate_id"],
            ascending=[True,True,True,True]
        )
    return out

stratified_screening = stratified_screening_table()
display(stratified_screening.head(60))

## 15. Per-query top-N queue

Every successful query/database execution contributes up to `PER_QUERY_SCREEN_N` records.

This is important because global recurrence is not a valid proxy for relevance. A niche but load-bearing search is guaranteed representation in the screening package.

In [ ]:
def per_query_top_n(n=25):
    if not len(active_retrievals):
        return pd.DataFrame()

    x = active_retrievals.merge(
        active_searches[
            ["search_id","query_plan_id","domain","search_purpose","database","query_used"]
        ],
        on=["search_id","query_plan_id","database"],
        how="left"
    ).merge(
        active_candidates[
            ["candidate_id","title","year","doi","screening_decision"]
        ],
        on="candidate_id",
        how="left"
    )
    x["_rank"] = pd.to_numeric(x["retrieval_rank"],errors="coerce")
    x = x.sort_values(
        ["query_plan_id","database","_rank"],
        ascending=[True,True,True],
        na_position="last"
    )
    out = x.groupby(
        ["query_plan_id","database"],
        as_index=False,
        group_keys=False
    ).head(n)
    keep = [
        "query_plan_id","domain","search_purpose","database",
        "query_used","retrieval_rank","candidate_id",
        "title","year","doi","screening_decision"
    ]
    return out[keep].reset_index(drop=True)

per_query_top = per_query_top_n(PER_QUERY_SCREEN_N)
display(per_query_top.head(60))

## 16. Dedicated evidence-role queues

In [ ]:
def role_queue(role):
    if not len(stratified_screening):
        return pd.DataFrame()
    return stratified_screening[
        stratified_screening["search_purpose"]==role
    ].copy()

direct_queue = role_queue("DIRECT")
counterevidence_queue = role_queue("COUNTEREVIDENCE")
foundational_queue = role_queue("FOUNDATIONAL")
recent_queue = role_queue("RECENT")
replication_boundary_queue = role_queue("REPLICATION_BOUNDARY")
methodological_queue = role_queue("METHODOLOGICAL")

cross_domain_queue = candidate_summary[
    pd.to_numeric(candidate_summary.get("domain_count",0),errors="coerce") >= 2
].copy() if len(candidate_summary) else pd.DataFrame()

print("Direct:",len(direct_queue))
print("Counterevidence:",len(counterevidence_queue))
print("Foundational:",len(foundational_queue))
print("Recent:",len(recent_queue))
print("Replication/boundary:",len(replication_boundary_queue))
print("Cross-domain:",len(cross_domain_queue))

## 17. Manual source-specific search queue

OpenAlex/PubMed are not treated as substitutes for ACM, IEEE, OSF, publishers or other source-specific databases.

Every non-automated source named in the frozen search plan becomes an explicit unresolved task.

In [ ]:
AUTOMATED_SOURCES = {"openalex","pubmed"}

def manual_source_queue_table():
    rows = []
    for _,r in plan.iterrows():
        sources = [
            x.strip()
            for x in str(r["preferred_sources"]).split(";")
            if x.strip()
        ]
        for src in sources:
            if src.lower() not in AUTOMATED_SOURCES:
                rows.append({
                    "query_plan_id":r["query_plan_id"],
                    "domain":r["domain"],
                    "search_purpose":r["search_purpose"],
                    "manual_source":src,
                    "planned_query":r["planned_query"],
                    "rationale":r["rationale"],
                    "status":"NOT_YET_RUN",
                    "results_screened":"",
                    "retained_candidate_ids":"",
                    "notes":""
                })
    return pd.DataFrame(rows)

manual_source_queue = manual_source_queue_table()
display(manual_source_queue.head(50))
print("Unresolved source-specific search tasks:",len(manual_source_queue))

## 18. Search-coverage diagnostics

In [ ]:
def search_coverage_table():
    rows = []
    for _,p in plan.iterrows():
        qid = p["query_plan_id"]
        rec = {
            "query_plan_id":qid,
            "domain":p["domain"],
            "search_purpose":p["search_purpose"],
            "planned_query":p["planned_query"]
        }
        for db in ["OpenAlex","PubMed"]:
            x = searches[
                (searches["query_plan_id"].astype(str)==qid) &
                (searches["database"].astype(str)==db)
            ].copy()
            if len(x):
                x["_ts"] = pd.to_datetime(x["run_timestamp"],errors="coerce")
                x = x.sort_values("_ts")
                latest = x.iloc[-1]
                rec[f"{db}_latest_status"] = latest["status"]
                rec[f"{db}_reported_total"] = latest["total_results_reported"]
                rec[f"{db}_retrieved"] = latest["results_retrieved"]
                rec[f"{db}_partial"] = latest["partial_retrieval"]
                rec[f"{db}_error"] = latest["error"]
            else:
                rec[f"{db}_latest_status"] = "NOT_RUN"
                rec[f"{db}_reported_total"] = ""
                rec[f"{db}_retrieved"] = ""
                rec[f"{db}_partial"] = ""
                rec[f"{db}_error"] = ""
        rows.append(rec)
    return pd.DataFrame(rows)

search_coverage = search_coverage_table()
display(search_coverage.head(40))

## 19. Fuzzy duplicate candidates — report only

Fuzzy matching is used only to identify **possible** duplicates for later inspection.

No rows are merged by this function.

In [ ]:
def fuzzy_duplicate_report(threshold=97):
    c = active_candidates.reset_index(drop=True)
    rows = []

    # Block by first normalized-title character and year to avoid O(N^2) explosion.
    c["_nt"] = c["title"].astype(str).map(normalize_title)
    c["_yr"] = c["year"].astype(str).map(clean_year)
    c["_block"] = c["_nt"].str[:1] + "|" + c["_yr"]

    for block,g in c.groupby("_block"):
        if len(g) < 2 or len(g) > 500:
            continue
        idxs = list(g.index)
        for a in range(len(idxs)):
            i = idxs[a]
            for b in range(a+1,len(idxs)):
                j = idxs[b]
                ti = c.loc[i,"_nt"]
                tj = c.loc[j,"_nt"]
                if not ti or not tj or ti == tj:
                    continue
                sim = fuzz.ratio(ti,tj)
                if sim >= threshold:
                    rows.append({
                        "candidate_1":c.loc[i,"candidate_id"],
                        "candidate_2":c.loc[j,"candidate_id"],
                        "year_1":c.loc[i,"year"],
                        "year_2":c.loc[j,"year"],
                        "title_similarity":sim,
                        "title_1":c.loc[i,"title"],
                        "title_2":c.loc[j,"title"],
                        "manual_duplicate_decision":""
                    })
    return pd.DataFrame(rows)

fuzzy_duplicates = fuzzy_duplicate_report()
print("Possible fuzzy duplicates:",len(fuzzy_duplicates))

## 20. Post-harvest integrity audit

This audit checks organization/provenance only.

It does **not** declare literature completeness.

It checks:
- both automated databases have an execution record for every planned query;
- retrievals point to valid searches and candidates;
- source records point to valid candidates;
- composite retrievals are not duplicated;
- new Step-1B candidates remain `NOT_SCREENED`;
- search counts are internally coherent;
- every successful search is represented in the per-query screening queue;
- large/partial searches are explicitly flagged;
- non-automated source tasks remain visible.

In [ ]:
def post_harvest_integrity_audit():
    issues = []
    search_ids = set(searches["search_id"].astype(str))
    candidate_ids = set(candidates["candidate_id"].astype(str))

    # Every query/database pair must have at least one execution record.
    for qid in plan["query_plan_id"]:
        for db in ["OpenAlex","PubMed"]:
            x = searches[
                (searches["query_plan_id"].astype(str)==str(qid)) &
                (searches["database"].astype(str)==db)
            ]
            if not len(x):
                issues.append({
                    "severity":"BLOCKING",
                    "type":"SEARCH_NOT_EXECUTED",
                    "target":f"{qid}|{db}",
                    "message":"No execution record."
                })
            elif not (x["status"]=="OK").any():
                issues.append({
                    "severity":"BLOCKING",
                    "type":"SEARCH_FAILED",
                    "target":f"{qid}|{db}",
                    "message":"No successful execution; rerun notebook."
                })

    # Retrieval foreign keys.
    for _,r in active_retrievals.iterrows():
        if str(r["search_id"]) not in search_ids:
            issues.append({
                "severity":"BLOCKING","type":"BROKEN_SEARCH_LINK",
                "target":r["retrieval_id"],"message":"Retrieval points to missing search."
            })
        if str(r["candidate_id"]) not in candidate_ids:
            issues.append({
                "severity":"BLOCKING","type":"BROKEN_CANDIDATE_LINK",
                "target":r["retrieval_id"],"message":"Retrieval points to missing candidate."
            })

    # Source-record foreign keys.
    for _,r in active_source_records.iterrows():
        if str(r["candidate_id"]) not in candidate_ids:
            issues.append({
                "severity":"BLOCKING","type":"BROKEN_SOURCE_CANDIDATE_LINK",
                "target":r["source_record_id"],"message":"Source record points to missing candidate."
            })

    # Duplicate retrieval composites.
    if len(active_retrievals):
        comp = active_retrievals[
            ["search_id","candidate_id","database_record_id"]
        ].astype(str).agg("|".join,axis=1)
        dups = active_retrievals[comp.duplicated(keep=False)]
        for _,r in dups.iterrows():
            issues.append({
                "severity":"BLOCKING","type":"DUPLICATE_RETRIEVAL",
                "target":r["retrieval_id"],"message":"Duplicate retrieval composite."
            })

    # New Step1B candidates may not be auto-screened.
    new_ids = set(candidate_origin.loc[
        candidate_origin["created_in_step1b"].astype(str)=="yes",
        "candidate_id"
    ].astype(str))
    if new_ids:
        x = candidates[candidates["candidate_id"].astype(str).isin(new_ids)]
        changed = x[x["screening_decision"].astype(str)!="NOT_SCREENED"]
        for _,r in changed.iterrows():
            issues.append({
                "severity":"BLOCKING","type":"AUTOMATIC_SCREENING",
                "target":r["candidate_id"],
                "message":"New Step1B candidate has a non-NOT_SCREENED decision."
            })

    # Report partial retrievals as coverage warnings, not logic failures.
    for _,r in active_searches.iterrows():
        try:
            total = int(float(r["total_results_reported"]))
            got = int(float(r["results_retrieved"]))
            if total > got:
                severity = "REVIEW" if total <= 1000 else "HIGH_VOLUME_REVIEW"
                issues.append({
                    "severity":severity,
                    "type":"PARTIAL_RETRIEVAL",
                    "target":f"{r['query_plan_id']}|{r['database']}",
                    "message":f"Retrieved {got} of {total} reported results."
                })
        except Exception:
            pass

    # Every successful search should contribute to per-query screening.
    top_search_pairs = set(
        zip(
            per_query_top["query_plan_id"].astype(str),
            per_query_top["database"].astype(str)
        )
    ) if len(per_query_top) else set()
    for _,r in active_searches.iterrows():
        pair = (str(r["query_plan_id"]),str(r["database"]))
        try:
            n = int(float(r["results_retrieved"]))
        except Exception:
            n = 0
        if n > 0 and pair not in top_search_pairs:
            issues.append({
                "severity":"BLOCKING","type":"MISSING_QUERY_STRATUM",
                "target":"|".join(pair),
                "message":"Successful search has retrieved records but no per-query screening stratum."
            })

    if len(manual_source_queue)==0:
        issues.append({
            "severity":"REVIEW","type":"NO_MANUAL_SOURCE_QUEUE",
            "target":"manual sources",
            "message":"No non-automated source tasks were generated; verify source parsing."
        })

    return pd.DataFrame(issues)

integrity_warnings = post_harvest_integrity_audit()
display(integrity_warnings.head(100))

blocking = integrity_warnings[
    integrity_warnings["severity"]=="BLOCKING"
] if len(integrity_warnings) else pd.DataFrame()

if len(blocking):
    print("\nPOST-HARVEST AUDIT: BLOCKING ISSUES PRESENT")
    print("Rerun/resolve these before treating the package as ready.")
else:
    print("\nPOST-HARVEST AUDIT: NO BLOCKING PROVENANCE ISSUES")
    print("Partial-retrieval/high-volume warnings still require later scientific review.")

## 21. Domain and role coverage summary

This is descriptive. `READY_FOR_SCIENTIFIC_SCREENING` means the automated search executions are present; it does **not** mean that literature saturation or scientific Step-1 closure has been established.

In [ ]:
def domain_coverage_summary():
    rows = []
    for domain,gp in plan.groupby("domain"):
        qids = set(gp["query_plan_id"].astype(str))
        s = active_searches[
            active_searches["query_plan_id"].astype(str).isin(qids)
        ]
        r = active_retrievals[
            active_retrievals["search_id"].astype(str).isin(
                set(s["search_id"].astype(str))
            )
        ]
        purposes = sorted(set(gp["search_purpose"].astype(str)))
        rows.append({
            "domain":domain,
            "planned_queries":len(gp),
            "planned_search_purposes":"; ".join(purposes),
            "successful_automated_executions":len(s),
            "expected_automated_executions":len(gp)*2,
            "unique_candidates":r["candidate_id"].nunique() if len(r) else 0,
            "automated_status":(
                "READY_FOR_SCIENTIFIC_SCREENING"
                if len(s)==len(gp)*2 else
                "INCOMPLETE_AUTOMATED_EXECUTION"
            )
        })
    return pd.DataFrame(rows)

domain_coverage = domain_coverage_summary()
display(domain_coverage)

## 22. Version manifest

This freezes the exact automated harvest configuration so later modifications cannot masquerade as the same Step-1B run.

In [ ]:
manifest = {
    "phase":"STEP1B",
    "generated_at":pd.Timestamp.now().isoformat(),
    "email_not_exported":True,
    "query_count":int(len(plan)),
    "domain_count":int(plan["domain"].nunique()),
    "base_plan_sha256":EXPECTED_BASE_HASH,
    "expanded_plan_sha256":EXPECTED_FULL_HASH,
    "max_per_query_per_database":MAX_PER_QUERY_PER_DATABASE,
    "recent_start_date":RECENT_START_DATE,
    "automated_databases":["OpenAlex","PubMed"],
    "successful_active_searches":int(len(active_searches)),
    "expected_active_searches":int(len(plan)*2),
    "unique_step1b_candidates":int(len(active_candidates)),
    "step1b_retrievals":int(len(active_retrievals)),
    "manual_source_tasks":int(len(manual_source_queue)),
    "blocking_integrity_issues":int(len(blocking))
}

MANIFEST_PATH = STEP1B / "Step1B_Manifest.json"
MANIFEST_PATH.write_text(
    json.dumps(manifest,indent=2),
    encoding="utf-8"
)
print(json.dumps(manifest,indent=2))

## 23. Export the Step-1B screening package

The notebook saves:
- CSV mirrors for every major table;
- a ZIP containing those CSVs;
- a single Excel workbook for convenient upload/review.

The workbook is a **screening package**, not a completed evidence ledger. Primary-source verification, experiment/result extraction and scientific adjudication come next.

In [ ]:
OUTPUT = STEP1B / "Output"
OUTPUT.mkdir(parents=True, exist_ok=True)

exports = {
    "Query_Plan":plan,
    "Searches":active_searches,
    "Retrievals":active_retrievals,
    "Source_Records":active_source_records,
    "Candidate_Summary":candidate_summary,
    "Stratified_Screening":stratified_screening,
    "Per_Query_Top25":per_query_top,
    "Direct_Queue":direct_queue,
    "Counterevidence_Queue":counterevidence_queue,
    "Foundational_Queue":foundational_queue,
    "Recent_Queue":recent_queue,
    "Replication_Boundary":replication_boundary_queue,
    "Methodological_Queue":methodological_queue,
    "Cross_Domain":cross_domain_queue,
    "Search_Coverage":search_coverage,
    "Domain_Coverage":domain_coverage,
    "Manual_Source_Queue":manual_source_queue,
    "Fuzzy_Duplicates":fuzzy_duplicates,
    "Integrity_Warnings":integrity_warnings
}

csv_paths = []
for name,df in exports.items():
    path = OUTPUT / f"{name}.csv"
    df.to_csv(path,index=False)
    csv_paths.append(path)

# ZIP mirror
zip_path = STEP1B / "Step1B_Screening_Package_CSVs.zip"
with zipfile.ZipFile(zip_path,"w",zipfile.ZIP_DEFLATED) as z:
    for path in csv_paths:
        z.write(path,arcname=path.name)
    z.write(MANIFEST_PATH,arcname=MANIFEST_PATH.name)

# Single workbook
xlsx_path = STEP1B / "Step1B_Screening_Package.xlsx"
with pd.ExcelWriter(xlsx_path,engine="openpyxl") as w:
    for name,df in exports.items():
        sheet = name[:31]
        # Excel cell limit protection: truncate ultra-long abstracts only in workbook,
        # never in CSV/source data.
        out = df.copy()
        for col in out.columns:
            if out[col].dtype == object:
                out[col] = out[col].map(
                    lambda x: str(x)[:32000] if pd.notna(x) else ""
                )
        out.to_excel(w,sheet_name=sheet,index=False)

print("EXPORT COMPLETE")
print("Workbook:",xlsx_path)
print("CSV package:",zip_path)
print("\nUpload Step1B_Screening_Package.xlsx back into the chat.")

# What happens after this notebook

Do **not** independently decide which thousands of papers to keep.

Upload:

`Step1B_Screening_Package.xlsx`

The next stage is a scientific screening audit in chat. We will use:
- per-query strata;
- counterevidence strata;
- domain coverage;
- cross-domain bridges;
- manual-source gaps;
- abstracts and metadata;

to determine which candidate papers require primary-record verification and full-text/result extraction.

Only after that do we populate:
- verified publications;
- experiments;
- atomic results;
- evidence claims;

and then perform the actual Step-1 scientific adjudication.